# SpottingSalmon
YOLO model logging script
- To register the final model as an MLFlow model


In [0]:
%pip install --quiet numpy==1.26.4  mlflow ultralytics

In [0]:
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.pyfunc
import ultralytics
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, TensorSpec, DataType, ColSpec
import numpy as np
from ultralytics import YOLO
from pyspark.sql import functions as F

### 1. Model logging

In [0]:
# Run ID and artifact relative path
run_id = "cc1ca4e121884869ac75efcba1a0af4a"
artifact_path = "weights/best.pt"

# Download to local temp dir
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path=artifact_path
)

print(f"Model downloaded to: {model_path}")

# Load the model
model = YOLO(model_path)

In [0]:
#%% ---------------------------------------------
# Full cell: Predict + Annotated Video Output
#-----------------------------------------------
import mlflow.pyfunc
import pandas as pd
from ultralytics import YOLO
import cv2
import os
import shutil
import uuid
from databricks.sdk import WorkspaceClient

#################
# Single logger #
#################
def _log(msg: str):
    print(msg, flush=True)

###################
# UC file to temp #
###################
def download_uc_file_to_tmp(w: WorkspaceClient, uc_path: str) -> str:
    local_path = f"/tmp/{uuid.uuid4().hex}.mp4"
    _log(f"[INFO] Downloading UC file: {uc_path}")

    files_api = w.files

    if hasattr(files_api, "download_to"):
        files_api.download_to(uc_path, local_path)
    else:
        resp = files_api.download(uc_path)
        stream = getattr(resp, "contents", None)
        if stream is None:
            raise RuntimeError("Files API response missing `.contents` attribute.")
        with stream as f, open(local_path, "wb") as out:
            shutil.copyfileobj(f, out, length=1024 * 1024)

    size_mb = os.path.getsize(local_path) / (1024 * 1024)
    _log(f"[INFO] Download complete -> {local_path} ({size_mb:.2f} MB)")
    return local_path

###############
# Read frames #
###############
def read_video_frames(video_path: str, w: WorkspaceClient | None = None):
    _log(f"[INFO] Reading video: {video_path}")

    cleanup = False
    local_path = video_path

    if w is not None:
        local_path = download_uc_file_to_tmp(w, video_path)
        cleanup = True
    else:
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"Video path not accessible: {local_path}")

    cap = cv2.VideoCapture(local_path)
    if not cap.isOpened():
        raise RuntimeError(f"OpenCV could not open video: {local_path}")

    ret, first_frame = cap.read()
    if not ret:
        cap.release()
        if cleanup:
            try:
                os.remove(local_path)
            except Exception as e:
                _log(f"[WARN] Could not remove temp file {local_path}: {e}")
        raise RuntimeError(f"Empty or unreadable video: {local_path}")

    fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0)
    if fps <= 1.0:
        fps = 30.0

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0)
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0)
    if width <= 0 or height <= 0:
        height, width = first_frame.shape[:2]

    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    _log(f"[INFO] Video opened OK -> {local_path} (frames~{frame_count}, fps={fps:.2f}, size={width}x{height})")

    yield {"meta": True, "fps": fps, "width": width, "height": height, "local_path": local_path}

    frame_number = 0
    try:
        yield frame_number, first_frame
        frame_number += 1

        while True:
            ret, frame = cap.read()
            if not ret:
                _log(f"[INFO] End of video at frame {frame_number}")
                break
            yield frame_number, frame
            frame_number += 1
    finally:
        cap.release()
        if cleanup:
            try:
                os.remove(local_path)
                _log(f"[INFO] Temp file removed -> {local_path}")
            except Exception as e:
                _log(f"[WARN] Could not remove temp file {local_path}: {e}")

############################
# YOLO output to python df #
############################
def convert_yolo_to_dets(result):
    detections = []
    if result is None or result.boxes is None or len(result.boxes) == 0:
        return detections

    boxes = result.boxes
    xyxy = boxes.xyxy.cpu().numpy()
    confs = boxes.conf.cpu().numpy()

    for (x1, y1, x2, y2), conf in zip(xyxy, confs):
        detections.append({
            "x1": float(x1),
            "y1": float(y1),
            "x2": float(x2),
            "y2": float(y2),
            "confidence": float(conf)
        })
    return detections

#################
# Frame tracker #
#################
class SimpleFrameGapTracker:
    def __init__(self, max_gap=5):
        self.max_gap = max_gap
        self.last_frame_seen = None
        self.current_track_id = 1

    def update(self, frame_number, detections):
        if len(detections) == 0:
            return []

        if self.last_frame_seen is None:
            self.last_frame_seen = frame_number
        elif frame_number - self.last_frame_seen > self.max_gap:
            self.current_track_id += 1

        self.last_frame_seen = frame_number

        output = []
        for d in detections:
            output.append({"track_id": self.current_track_id, **d})
        return output

#########################
# Draw overlay on frame #
#########################
def draw_boxes(frame, detections):
    for det in detections:
        x1, y1, x2, y2 = map(int, [det["x1"], det["y1"], det["x2"], det["y2"]])
        track_id = det["track_id"]
        conf = det.get("confidence", 0.0)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (128, 0, 0), 2)

        label = f"Fish # {track_id} ({conf:.2f})"
        cv2.putText(
            frame,
            label,
            (x1, max(0, y1 - 8)),
            cv2.FONT_HERSHEY_TRIPLEX,
            0.5,
            (255, 0, 0),
            1,
            cv2.LINE_AA
        )
    return frame

# ------------------------------
# YOLO + Simple Tracker PyFunc
# ------------------------------
class FishVideoDetector(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        _log("[INFO] Loading YOLO model")
        self.model = YOLO(context.artifacts["checkpoint"])

        _log("[INFO] Initializing SimpleFrameGapTracker")
        self.tracker = SimpleFrameGapTracker(max_gap=5)

        host = os.environ.get("DATABRICKS_HOST")
        token = os.environ.get("DATABRICKS_TOKEN")
        self.w = WorkspaceClient(host=host, token=token) if host and token else None

    def predict(self, context, model_input: pd.DataFrame):
        if "fish" not in model_input:
            return pd.DataFrame([])

        results = []

        for idx, video_path in enumerate(model_input["fish"]):
            _log(f"[INFO] Processing video: {video_path}")

            self.tracker = SimpleFrameGapTracker(max_gap=5)

            try:
                video_name = os.path.basename(video_path)

                gen = read_video_frames(video_path, w=self.w)
                meta = next(gen)
                fps = float(meta["fps"])
                width = int(meta["width"])
                height = int(meta["height"])

                out_dir = "/tmp"
                os.makedirs(out_dir, exist_ok=True)
                output_path = os.path.join(
                    out_dir,
                    f"{os.path.splitext(video_name)[0]}_annotated_{uuid.uuid4().hex[:8]}.mp4"
                )

                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
                if not writer.isOpened():
                    raise RuntimeError("Failed to open cv2.VideoWriter.")

                for frame_number, frame in gen:
                    out = self.model(frame, verbose=False)
                    if isinstance(out, list):
                        out = out[0]

                    dets = convert_yolo_to_dets(out)
                    tracked = self.tracker.update(frame_number, dets)

                    for t in tracked:
                        results.append({
                            "track_id": t["track_id"],
                            "video": video_name,
                            "frame": frame_number,
                            "x1": t["x1"],
                            "y1": t["y1"],
                            "x2": t["x2"],
                            "y2": t["y2"],
                            "confidence": t.get("confidence", 0.0),
                            "annotated_video_path": output_path
                        })

                    annotated = draw_boxes(frame.copy(), tracked)
                    writer.write(annotated)

                writer.release()
                _log(f"[INFO] Annotated video saved: {output_path}")

            except StopIteration:
                _log(f"[ERROR] No frames available in video.")
            except Exception as e:
                _log(f"[ERROR] failed video {video_path}: {e}")

        return pd.DataFrame(results)

In [0]:
import pandas as pd
from mlflow.models.signature import ModelSignature
from mlflow.types import Schema, ColSpec

input_example = pd.DataFrame({
    "fish": ["/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"]
})


signature = ModelSignature(
    inputs=Schema([ColSpec("string", "fish")]),
    outputs=Schema([
        ColSpec("integer", "fish_id"),
        ColSpec("string", "video"),
        ColSpec("integer", "frame"),
        ColSpec("float", "x1"),
        ColSpec("float", "y1"),
        ColSpec("float", "x2"),
        ColSpec("float", "y2"),
        ColSpec("float", "confidence"), 
        ColSpec("string", "annotated_video_path")
    ])
)

In [0]:
with mlflow.start_run() as run:
    run_id = run.info.run_id

    mlflow.pyfunc.log_model(
        name="model",  # 
        python_model=FishVideoDetector(),
        input_example=input_example,
        artifacts={"checkpoint": model_path},
        signature=signature
    )

print("Logged model under run:", run_id)


### Register model

In [0]:
import mlflow

mlflow.set_registry_uri("databricks")

model_uri = f"runs:/{run_id}/model"

print("About to register model from:")
print("Run ID:", run_id)
print("Model URI:", model_uri)

result = mlflow.register_model(
    model_uri=model_uri,
    name="FishVideoDetector"
)

print("\nRegistered model version:", result.version)
print("Registered from run ID:", result.run_id)

# Extra safety check
if result.run_id == run_id:
    print("Registration confirmed: correct run ID used.")
else:
    print(" WARNING: Registered run ID does NOT match expected run ID!")



In [0]:
import mlflow.pyfunc
import pandas as pd

mlflow.set_registry_uri("databricks")

model_uri = "models:/FishVideoDetector/18"
model = mlflow.pyfunc.load_model(model_uri)

input_df = pd.DataFrame({
    "fish": [
        "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/4_2021-07-06_14-11-37x.mp4"
    ]
})

preds = model.predict(input_df)
print(preds)


In [0]:
display(preds)

In [0]:

import shutil, os

src = "/tmp/4_2021-07-06_14-11-37x_annotated_da845d7b.mp4"

# Replace with your actual workspace login email
workspace_dir = "/Volumes/prd_dash_lab/dash_data_science_unrestricted/shared_external_volume/rachels_stuff/"

dst = f"{workspace_dir}/{os.path.basename(src)}"
shutil.copy(src, dst)

print("Saved to:", dst)
